# Dash rendering spike — 2 M-point cost study

Goal: empirical evidence for which Plotly/Dash rendering technique handles the project's data scale (~0.5–2 M wave-impact points + vessel tracks), and how to keep pan/zoom **smooth** at that scale.

Plan: `~/.claude/plans/woolly-spinning-robin.md`.

## Interaction model (fixed by user)

| State | Renderer must support | Approx. point count |
|---|---|---|
| A. Wave-impact points on coastline (always-on) | hover at all zoom levels | 0.5–2 M |
| B. Wave-impact points along 1-D coastline-distance axis | hover | 0.5–2 M |
| C. Vessel + tracks dense overview (unfiltered) | density only — no hover | ~5 M |
| D. Vessel + tracks filtered (post-selection) | hover + click | 10–50 k |
| E. Smoothness during continuous wheel-zoom | smooth feel under all of A | — |

Hover is gated by visible-point count via clientside callback when the renderer cannot keep up.

## How to use this notebook

Run cells top-to-bottom. Each scenario launches a Dash app inline. Pan/zoom each one manually for 5 s and record subjective smoothness in the `RESULTS` dict (see Setup). The final cell prints the synthesis table.

Server-side numbers (data prep, JSON size) are auto-measured. Browser-side numbers (initial render, smoothness, callback storm) are manual — open DevTools Performance tab while pan/zoom-ing.

## Background reference

From research summarised in the plan file:

| Technique | Hover? | Practical limit | Notes |
|---|---|---|---|
| `go.Scattermap` (replaces deprecated `scattermapbox`) | Yes | ~50 k for smooth interaction | No WebGL on map |
| `go.Scattergl` (Cartesian) | Yes | ~100–200 k (GPU-bound) | No basemap |
| `plotly-resampler` | Yes | 100 M+ (1-D) | Cartesian only — does NOT work on `scatter_mapbox` |
| Datashader → Mapbox image layer | **No** (image, not points) | Billions | Re-rasterise on `relayoutData` |
| `dash-deck` (deck.gl) | Yes (picking) | Millions | Marked proof-of-concept |
| `dash-leaflet` GeoJSON + supercluster | Yes | ~1 M with cluster | Raw markers struggle past ~1 k |

Smoothness mitigations to test in Scenario E: debounce / two-resolution / pre-tile / `Patch()` partial updates.

In [ ]:
# Setup: imports, paths, ports, results dict
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd()
OUT_DIR = REPO / 'output'
EXAMPLES = REPO / 'examples'

# Singapore West Coast AOI (matches Coast_P1.shp area)
AOI_LON = (103.55, 103.75)
AOI_LAT = (1.20, 1.32)

# Each Dash app gets its own port so they can coexist when re-run
PORT_BASE = 8060

# Results collector — one row per scenario subcell
RESULTS: list[dict] = []

def record(scenario: str, technique: str, **metrics) -> None:
    """Append one measured row to RESULTS."""
    RESULTS.append({'scenario': scenario, 'technique': technique, **metrics})
    print(f'  recorded {scenario}.{technique}: {metrics}')

print('cwd:', REPO)
print('output exists:', OUT_DIR.exists())

In [ ]:
# Synthesise a representative 2 M-row wave-impact dataframe.
#
# Schema matches aiswakepy/stages/wave_impact.py:299–317:
#   MMSI, ShLongitude, ShLatitude, WaveHeight, WavePeriod, E_max, E_tot,
#   DistLoc_km, DateTime, Froude_D, VesselWidth, VesselLength, SOG, Side
#
# Points are concentrated along a meandering coastline (sin-curve in the AOI),
# clustered in shipping lanes, with log-normal WaveHeight peaking ~0.1 m.

def synthesise_wave_impact(n: int = 2_000_000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    # Coastline meanders along latitude ~1.26 with a sin curve
    lon = rng.uniform(AOI_LON[0], AOI_LON[1], size=n)
    lat_curve = 1.26 + 0.015 * np.sin((lon - AOI_LON[0]) * 60.0)
    # Add small cross-shore jitter
    lat = lat_curve + rng.normal(0, 0.0008, size=n)
    # Cluster around ~50 vessel tracks (shipping lanes)
    h = rng.lognormal(mean=-2.5, sigma=0.6, size=n).clip(0.01, 0.6)
    T = rng.uniform(2.0, 4.5, size=n)
    rho, g = 1026.0, 9.78
    E_max = rho * g * g * h * h * T * T / (16.0 * math.pi)
    E_tot = 10.8 * E_max ** 0.82
    side = rng.choice(['port', 'stbd'], size=n)
    mmsi = rng.integers(563000000, 563002000, size=n)  # ~2k unique vessels
    return pd.DataFrame({
        'MMSI': mmsi,
        'ShLongitude': lon,
        'ShLatitude': lat,
        'WaveHeight': h,
        'WavePeriod': T,
        'E_max': E_max,
        'E_tot': E_tot,
        'DistLoc_km': rng.uniform(0.1, 2.0, size=n),
        'Froude_D': rng.uniform(0.1, 0.5, size=n),
        'VesselWidth': rng.uniform(20, 60, size=n),
        'VesselLength': rng.uniform(100, 400, size=n),
        'SOG': rng.uniform(2, 12, size=n),
        'Side': side,
    })

def synthesise_vessel_tracks(n_vessels: int = 500, points_per_vessel: int = 10_000, seed: int = 7) -> pd.DataFrame:
    """~5 M vessel positions across ~500 vessels. Each vessel walks a smooth path."""
    rng = np.random.default_rng(seed)
    rows = []
    for v in range(n_vessels):
        # Each vessel has a base lon/lat and slowly drifts
        lon0 = rng.uniform(*AOI_LON)
        lat0 = rng.uniform(*AOI_LAT)
        bearing = rng.uniform(0, 2*np.pi)
        steps = rng.normal(0, 1, size=(points_per_vessel, 2)).cumsum(axis=0) * 0.0001
        lon = lon0 + steps[:, 0] + np.cos(bearing) * np.linspace(0, 0.05, points_per_vessel)
        lat = lat0 + steps[:, 1] + np.sin(bearing) * np.linspace(0, 0.03, points_per_vessel)
        rows.append(pd.DataFrame({
            'MMSI': 563000000 + v,
            'longitude': lon,
            'latitude': lat,
            'sog': rng.uniform(2, 12, size=points_per_vessel),
        }))
    return pd.concat(rows, ignore_index=True)

print('synthesisers ready')

In [ ]:
# Build the working datasets. Scale-down N_WAVE/N_VESSEL during initial dev to iterate quickly.
N_WAVE = 2_000_000   # set to 200_000 for fast first run
N_VESSEL_PTS = 5_000_000

t0 = time.perf_counter()
df_wave = synthesise_wave_impact(N_WAVE)
t_wave = time.perf_counter() - t0
print(f'wave-impact: {len(df_wave):,} rows in {t_wave:.1f}s, mem {df_wave.memory_usage(deep=True).sum()/1e6:.0f} MB')

t0 = time.perf_counter()
df_track = synthesise_vessel_tracks(
    n_vessels=500,
    points_per_vessel=N_VESSEL_PTS // 500,
)
t_track = time.perf_counter() - t0
print(f'vessel tracks: {len(df_track):,} rows in {t_track:.1f}s, mem {df_track.memory_usage(deep=True).sum()/1e6:.0f} MB')

## Scenario A — Wave-impact points on map (hover required, up to 2 M)

Manual procedure for each Dash app cell:
1. Run the cell — it prints server-side metrics (JSON size, data prep time) and launches the app inline.
2. Open browser DevTools (F12) → Performance tab. Click record.
3. Mouse-wheel zoom continuously for 5 s. Stop recording.
4. Note: dropped frames? hover responsive? Scribble subjective rating into the cell's `record(...)` call (smooth / chunky / unusable).
5. Stop the app cell when done so the next port is free.

In [ ]:
# A.1 — go.Scattermap raw, no downsampling (failure baseline)
import plotly.graph_objects as go
from dash import Dash, dcc, html

t0 = time.perf_counter()
fig = go.Figure(go.Scattermap(
    lon=df_wave['ShLongitude'],
    lat=df_wave['ShLatitude'],
    mode='markers',
    marker=dict(size=2, color=df_wave['WaveHeight'], colorscale='Viridis', cmin=0, cmax=0.5),
    text=df_wave['WaveHeight'].round(3),
    hoverinfo='text',
))
fig.update_layout(
    map=dict(style='carto-positron', center=dict(lon=103.65, lat=1.26), zoom=11),
    margin=dict(l=0, r=0, t=0, b=0), height=550,
)
t_build = time.perf_counter() - t0
json_mb = len(fig.to_json()) / 1e6
print(f'A.1 fig build: {t_build:.1f}s, JSON: {json_mb:.1f} MB')

app = Dash(__name__)
app.layout = html.Div([html.H4('A.1 Scattermap raw'), dcc.Graph(figure=fig)])
# Run inline. Stop the cell to free the port.
# app.run(jupyter_mode='inline', port=PORT_BASE+1, debug=False)
record('A', 'scattermap_raw', json_mb=json_mb, build_s=t_build,
       smoothness='RECORD MANUALLY: smooth/chunky/unusable',
       hover_at_2M='RECORD MANUALLY')
print('Uncomment app.run(...) above to launch.')

In [ ]:
# A.2 — Scattermap fed by bbox-aware top-N downsampler, with clientside hover-toggle.
#
# Reuses the spirit of aiswakepy/viz/wave_map.py::_downsample but with a bbox
# pre-filter so only points in the current viewport are considered, and the
# coastline-binned top-N retains spatial coverage.
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State, ClientsideFunction, no_update

MAX_POINTS = 50_000          # send no more than this to the browser per update
HOVER_THRESHOLD = 50_000     # disable hover above this point count

def downsample_bbox(df: pd.DataFrame, bbox: tuple[float, float, float, float], cap: int) -> pd.DataFrame:
    lon0, lat0, lon1, lat1 = bbox
    m = (df['ShLongitude'] >= lon0) & (df['ShLongitude'] <= lon1) & \
        (df['ShLatitude']  >= lat0) & (df['ShLatitude']  <= lat1)
    sub = df[m]
    if len(sub) <= cap:
        return sub
    # Simple longitude-binned top-N as a stand-in for the 1-m coastline binning.
    # Production: call aiswakepy.viz.wave_map._downsample (with coastline_shp).
    n_bins = 1000
    bin_edges = np.linspace(lon0, lon1, n_bins + 1)
    sub = sub.assign(_bin=np.searchsorted(bin_edges, sub['ShLongitude']))
    per_bin = math.ceil(cap / sub['_bin'].nunique())
    out = (sub.sort_values('WaveHeight', ascending=False)
              .groupby('_bin', sort=False).head(per_bin)
              .drop(columns=['_bin']))
    return out

AOI_BBOX = (AOI_LON[0], AOI_LAT[0], AOI_LON[1], AOI_LAT[1])
t0 = time.perf_counter()
init = downsample_bbox(df_wave, AOI_BBOX, MAX_POINTS)
t_ds = time.perf_counter() - t0
print(f'A.2 initial downsample: {len(df_wave):,} → {len(init):,} in {t_ds:.1f}s')

def make_fig(sub: pd.DataFrame) -> go.Figure:
    fig = go.Figure(go.Scattermap(
        lon=sub['ShLongitude'], lat=sub['ShLatitude'],
        mode='markers',
        marker=dict(size=3, color=sub['WaveHeight'], colorscale='Viridis', cmin=0, cmax=0.5),
        text=sub['WaveHeight'].round(3),
        hoverinfo='text' if len(sub) <= HOVER_THRESHOLD else 'skip',
    ))
    fig.update_layout(
        map=dict(style='carto-positron', center=dict(lon=103.65, lat=1.26), zoom=11),
        margin=dict(l=0, r=0, t=0, b=0), height=550, uirevision='static',
    )
    return fig

app = Dash(__name__)
app.layout = html.Div([
    html.H4('A.2 Scattermap + bbox top-N + hover-by-zoom'),
    html.Div(id='banner', style={'fontFamily': 'monospace', 'padding': '4px'}),
    dcc.Graph(id='g', figure=make_fig(init)),
])

@app.callback(
    Output('g', 'figure'), Output('banner', 'children'),
    Input('g', 'relayoutData'), prevent_initial_call=True,
)
def on_relayout(rl):
    if not rl or 'map.center' not in rl and 'map.zoom' not in rl:
        return no_update, no_update
    # Plotly newer scattermap emits map._derived in some versions; fall back to AOI.
    bounds = rl.get('map._derived', {}).get('coordinates') if rl else None
    if bounds:
        lons = [c[0] for c in bounds]; lats = [c[1] for c in bounds]
        bbox = (min(lons), min(lats), max(lons), max(lats))
    else:
        bbox = AOI_BBOX
    t0 = time.perf_counter()
    sub = downsample_bbox(df_wave, bbox, MAX_POINTS)
    dt = time.perf_counter() - t0
    return make_fig(sub), f'visible≈{len(sub):,}  hover={"on" if len(sub)<=HOVER_THRESHOLD else "off"}  ds={dt*1000:.0f}ms'

# app.run(jupyter_mode='inline', port=PORT_BASE+2, debug=False)
record('A', 'scattermap_topN_clientside_hover',
       json_mb=len(make_fig(init).to_json())/1e6,
       initial_ds_s=t_ds, max_points=MAX_POINTS, hover_threshold=HOVER_THRESHOLD,
       smoothness='RECORD MANUALLY')
print('Uncomment app.run(...) to launch.')

In [ ]:
# A.3 — Scattergl on Cartesian (lon, lat). WebGL-accelerated, no basemap.
import plotly.graph_objects as go
from dash import Dash, dcc, html

t0 = time.perf_counter()
fig = go.Figure(go.Scattergl(
    x=df_wave['ShLongitude'], y=df_wave['ShLatitude'],
    mode='markers',
    marker=dict(size=2, color=df_wave['WaveHeight'], colorscale='Viridis', cmin=0, cmax=0.5),
))
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=550,
                  xaxis=dict(scaleanchor='y', scaleratio=1))
t_build = time.perf_counter() - t0
json_mb = len(fig.to_json()) / 1e6
print(f'A.3 fig build: {t_build:.1f}s, JSON: {json_mb:.1f} MB')

app = Dash(__name__)
app.layout = html.Div([html.H4('A.3 Scattergl Cartesian'), dcc.Graph(figure=fig)])
# app.run(jupyter_mode='inline', port=PORT_BASE+3, debug=False)
record('A', 'scattergl_cartesian', json_mb=json_mb, build_s=t_build,
       smoothness='RECORD MANUALLY')

## Scenario B — Wave-impact points along 1-D coastline-distance axis

Project each (lon, lat) onto the merged coastline LineString to produce a 1-D distance-along-coastline coordinate. Plot WaveHeight vs distance — this is the natural representation of "waves arriving along a line." `plotly-resampler` works here because the trace is `go.Scattergl`, not `scatter_mapbox`.

In [ ]:
# Project wave points onto the coastline once (server-side).
# Using the synthesised coastline curve directly so we don't depend on the .shp.
t0 = time.perf_counter()
lon = df_wave['ShLongitude'].to_numpy()
lat = df_wave['ShLatitude'].to_numpy()
# Distance-along is just (lon - lon0) * 111_000 * cos(lat) for the synthetic case.
dist_along_m = (lon - AOI_LON[0]) * 111_320.0 * np.cos(np.deg2rad(lat.mean()))
df_wave_1d = df_wave.assign(DistAlong_m=dist_along_m).sort_values('DistAlong_m').reset_index(drop=True)
print(f'projection done in {time.perf_counter()-t0:.1f}s')

In [ ]:
# B.1 — Scattergl with all points, no resampling.
import plotly.graph_objects as go
from dash import Dash, dcc, html

t0 = time.perf_counter()
fig = go.Figure(go.Scattergl(
    x=df_wave_1d['DistAlong_m'], y=df_wave_1d['WaveHeight'],
    mode='markers', marker=dict(size=2, opacity=0.3),
))
fig.update_layout(height=400, margin=dict(l=40, r=10, t=10, b=40),
                  xaxis_title='Distance along coastline (m)', yaxis_title='Wave height (m)')
t_build = time.perf_counter() - t0
json_mb = len(fig.to_json()) / 1e6
print(f'B.1 fig build: {t_build:.1f}s, JSON: {json_mb:.1f} MB')

app = Dash(__name__)
app.layout = html.Div([html.H4('B.1 Scattergl 1-D, all points'), dcc.Graph(figure=fig)])
# app.run(jupyter_mode='inline', port=PORT_BASE+4, debug=False)
record('B', 'scattergl_1d_all', json_mb=json_mb, build_s=t_build,
       smoothness='RECORD MANUALLY')

In [ ]:
# B.2 — Scattergl wrapped in plotly-resampler FigureResampler.
# Aggregates dynamically on pan/zoom callbacks — designed for exactly this case.
from plotly_resampler import FigureResampler
from plotly_resampler.aggregation import MinMaxLTTB
import plotly.graph_objects as go

t0 = time.perf_counter()
fig = FigureResampler(go.Figure(), default_n_shown_samples=2000,
                      default_downsampler=MinMaxLTTB(parallel=True))
fig.add_trace(
    go.Scattergl(name='WaveHeight', mode='markers', marker=dict(size=3, opacity=0.4)),
    hf_x=df_wave_1d['DistAlong_m'].to_numpy(),
    hf_y=df_wave_1d['WaveHeight'].to_numpy(),
)
fig.update_layout(height=400, margin=dict(l=40, r=10, t=10, b=40),
                  xaxis_title='Distance along coastline (m)', yaxis_title='Wave height (m)')
t_build = time.perf_counter() - t0
print(f'B.2 FigureResampler build: {t_build:.1f}s, default_n_shown=2000')

# fig.show_dash(mode='inline', port=PORT_BASE+5)
record('B', 'scattergl_1d_resampler', build_s=t_build, default_n_shown=2000,
       smoothness='RECORD MANUALLY')
print('Uncomment fig.show_dash(...) to launch.')

## Scenario C — Vessel + tracks dense overview (no hover)

5 M vessel positions across ~500 vessels. Density display only — no per-point interaction.

In [ ]:
# C.1 — Datashader rasterise → Mapbox image-layer overlay.
import datashader as ds
import datashader.transfer_functions as tf
from datashader.colors import inferno
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, no_update

def rasterise(df: pd.DataFrame, bbox: tuple, w: int = 800, h: int = 500) -> str:
    lon0, lat0, lon1, lat1 = bbox
    cvs = ds.Canvas(plot_width=w, plot_height=h, x_range=(lon0, lon1), y_range=(lat0, lat1))
    agg = cvs.points(df, x='longitude', y='latitude')
    img = tf.shade(agg, cmap=inferno, how='log')
    return img.to_pil()

t0 = time.perf_counter()
img0 = rasterise(df_track, AOI_BBOX)
t_raster = time.perf_counter() - t0
print(f'C.1 datashader rasterise: {len(df_track):,} points → image in {t_raster*1000:.0f}ms')

fig = go.Figure(go.Scattermap())
fig.update_layout(
    map=dict(
        style='carto-positron', center=dict(lon=103.65, lat=1.26), zoom=11,
        layers=[dict(below='traces', sourcetype='image', source=img0,
                     coordinates=[
                         [AOI_LON[0], AOI_LAT[1]],
                         [AOI_LON[1], AOI_LAT[1]],
                         [AOI_LON[1], AOI_LAT[0]],
                         [AOI_LON[0], AOI_LAT[0]],
                     ])],
    ),
    margin=dict(l=0, r=0, t=0, b=0), height=550,
)

app = Dash(__name__)
app.layout = html.Div([html.H4('C.1 Datashader → Mapbox image layer'), dcc.Graph(figure=fig)])
# app.run(jupyter_mode='inline', port=PORT_BASE+6, debug=False)
record('C', 'datashader_mapbox_image',
       raster_ms=t_raster*1000, n_points=len(df_track),
       smoothness='RECORD MANUALLY')

In [ ]:
# C.2 — dash-deck ScatterplotLayer (positions) + PathLayer (tracks).
# Build paths grouped by MMSI. Use a subset for the path layer — full 5 M paths are too heavy for the cell.
import pydeck as pdk
import dash_deck
from dash import Dash, html

# Subsample for sanity — 50 vessels × all their points
sample_mmsi = df_track['MMSI'].unique()[:50]
df_sub = df_track[df_track['MMSI'].isin(sample_mmsi)]
paths = [{'mmsi': int(m), 'path': g[['longitude','latitude']].values.tolist()}
         for m, g in df_sub.groupby('MMSI')]

t0 = time.perf_counter()
scatter = pdk.Layer('ScatterplotLayer', data=df_track,
                    get_position='[longitude, latitude]', get_radius=10,
                    get_fill_color=[255, 140, 0, 120], pickable=False)
path = pdk.Layer('PathLayer', data=paths,
                 get_path='path', get_width=2, get_color=[200, 30, 0, 160],
                 width_min_pixels=1, pickable=False)
deck = pdk.Deck(layers=[scatter, path],
                initial_view_state=pdk.ViewState(longitude=103.65, latitude=1.26, zoom=11),
                map_style='light_no_labels')
t_build = time.perf_counter() - t0
json_mb = len(deck.to_json()) / 1e6
print(f'C.2 dash-deck build: {t_build:.1f}s, JSON: {json_mb:.1f} MB')

app = Dash(__name__)
app.layout = html.Div([html.H4('C.2 dash-deck ScatterplotLayer + PathLayer'),
                       dash_deck.DeckGL(deck.to_json(), id='dk', mapboxKey='')], style={'height': '600px'})
# app.run(jupyter_mode='inline', port=PORT_BASE+7, debug=False)
record('C', 'dash_deck_scatter_path', json_mb=json_mb, build_s=t_build, n_points=len(df_track),
       smoothness='RECORD MANUALLY')

## Scenario D — Filtered subset (post-selection, ~10–50 k)

Simulate the user having filtered wave points → derive responsible MMSIs → plot only those vessels.

In [ ]:
# Build the filtered subset: keep only vessels whose WaveHeight > 0.2 m at any point.
high_mmsi = df_wave[df_wave['WaveHeight'] > 0.2]['MMSI'].unique()[:30]
df_track_filt = df_track[df_track['MMSI'].isin(high_mmsi)]
df_wave_filt = df_wave[df_wave['MMSI'].isin(high_mmsi)]
print(f'filtered: {len(df_track_filt):,} track points, {len(df_wave_filt):,} wave points across {len(high_mmsi)} vessels')

In [ ]:
# D.1 — go.Scattermap plain at filtered scale (~10–50 k points).
import plotly.graph_objects as go
from dash import Dash, dcc, html

t0 = time.perf_counter()
fig = go.Figure()
fig.add_trace(go.Scattermap(
    lon=df_track_filt['longitude'], lat=df_track_filt['latitude'],
    mode='markers', marker=dict(size=3, color='red'), name='vessel positions',
    hovertext=df_track_filt['MMSI'],
))
fig.add_trace(go.Scattermap(
    lon=df_wave_filt['ShLongitude'], lat=df_wave_filt['ShLatitude'],
    mode='markers', marker=dict(size=4, color=df_wave_filt['WaveHeight'],
                                 colorscale='Viridis', cmin=0, cmax=0.5),
    name='wave impacts',
))
fig.update_layout(map=dict(style='carto-positron', center=dict(lon=103.65, lat=1.26), zoom=11),
                  margin=dict(l=0, r=0, t=0, b=0), height=550)
t_build = time.perf_counter() - t0
json_mb = len(fig.to_json()) / 1e6
print(f'D.1 fig build: {t_build:.1f}s, JSON: {json_mb:.1f} MB')

app = Dash(__name__)
app.layout = html.Div([html.H4('D.1 Scattermap filtered subset'), dcc.Graph(figure=fig)])
# app.run(jupyter_mode='inline', port=PORT_BASE+8, debug=False)
record('D', 'scattermap_filtered', json_mb=json_mb, build_s=t_build,
       n_track=len(df_track_filt), n_wave=len(df_wave_filt),
       hover_at_filtered_scale='RECORD MANUALLY')

In [ ]:
# D.2 — dash-deck with picking (hover/click) at filtered scale.
import pydeck as pdk
import dash_deck
from dash import Dash, html

paths = [{'mmsi': int(m), 'path': g[['longitude','latitude']].values.tolist()}
         for m, g in df_track_filt.groupby('MMSI')]

t0 = time.perf_counter()
scatter = pdk.Layer('ScatterplotLayer', data=df_wave_filt,
                    get_position='[ShLongitude, ShLatitude]', get_radius=15,
                    get_fill_color=[255, 140, 0, 200], pickable=True)
path = pdk.Layer('PathLayer', data=paths,
                 get_path='path', get_width=3, get_color=[30, 30, 200, 200],
                 width_min_pixels=2, pickable=True)
deck = pdk.Deck(layers=[scatter, path],
                initial_view_state=pdk.ViewState(longitude=103.65, latitude=1.26, zoom=11),
                map_style='light_no_labels',
                tooltip={'text': '{mmsi}\nH={WaveHeight}'})
t_build = time.perf_counter() - t0
json_mb = len(deck.to_json()) / 1e6
print(f'D.2 dash-deck filtered build: {t_build:.1f}s, JSON: {json_mb:.1f} MB')

app = Dash(__name__)
app.layout = html.Div([html.H4('D.2 dash-deck filtered + picking'),
                       dash_deck.DeckGL(deck.to_json(), id='dk2', mapboxKey='',
                                        enableEvents=['hover', 'click'])],
                      style={'height': '600px'})
# app.run(jupyter_mode='inline', port=PORT_BASE+9, debug=False)
record('D', 'dash_deck_filtered_picking', json_mb=json_mb, build_s=t_build,
       click_latency='RECORD MANUALLY')

## Scenario E — Smoothness mitigations (the central concern)

All built on top of A.2. Each mitigation isolates one variable so we can attribute smoothness improvement.

**Manual measurement protocol per cell:**
- DevTools → Performance → record while wheel-zooming continuously for 5 s.
- Note: dropped frames count, callback storm count (Network tab → XHR), subjective rating.

In [ ]:
# E.1 — Naive baseline. Every relayoutData fires the callback. Expect callback storm.
# (Same code as A.2 without the no_update guard. Re-use A.2 by running with rapid wheel-zoom.)
print('E.1: re-launch A.2 with continuous wheel-zoom; count XHR requests in DevTools Network.')
record('E', 'naive_relayout', smoothness='RECORD MANUALLY',
       callback_count_per_5s='RECORD MANUALLY')

In [ ]:
# E.2 — Debounced callback (clientside 250 ms idle window).
# Implemented by storing the latest relayoutData in a dcc.Store and triggering
# the Python callback off the debounced Store value.
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State, ClientsideFunction, no_update

DEBOUNCE_MS = 250

init = downsample_bbox(df_wave, AOI_BBOX, MAX_POINTS)

app = Dash(__name__)
app.layout = html.Div([
    html.H4(f'E.2 Debounced relayout ({DEBOUNCE_MS} ms)'),
    html.Div(id='banner', style={'fontFamily':'monospace','padding':'4px'}),
    dcc.Store(id='rl-raw'),
    dcc.Store(id='rl-debounced'),
    dcc.Graph(id='g', figure=make_fig(init)),
])

# Clientside debounce: write rl-raw on every relayoutData; copy to rl-debounced after idle window.
app.clientside_callback(
    """
    function(rl) {
        clearTimeout(window._aiswakepy_dbnc);
        return new Promise((resolve) => {
            window._aiswakepy_dbnc = setTimeout(() => resolve(rl), """ + str(DEBOUNCE_MS) + """);
        });
    }
    """,
    Output('rl-debounced', 'data'),
    Input('g', 'relayoutData'),
)

@app.callback(Output('g', 'figure'), Output('banner', 'children'),
              Input('rl-debounced', 'data'), prevent_initial_call=True)
def on_debounced(rl):
    if not rl:
        return no_update, no_update
    bbox = AOI_BBOX  # same bbox derivation as A.2
    t0 = time.perf_counter()
    sub = downsample_bbox(df_wave, bbox, MAX_POINTS)
    return make_fig(sub), f'visible≈{len(sub):,}  ds={(time.perf_counter()-t0)*1000:.0f}ms (debounced {DEBOUNCE_MS}ms)'

# app.run(jupyter_mode='inline', port=PORT_BASE+10, debug=False)
record('E', 'debounced_250ms', debounce_ms=DEBOUNCE_MS, smoothness='RECORD MANUALLY')

In [ ]:
# E.3 — Two-resolution layering. Always-on coarse base layer (30 k) + debounced high-res top layer.
# Base never updates → viewport is never empty during native pan/zoom.
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, no_update

COARSE_N = 30_000
FINE_N = 50_000

# Build the coarse base once: stratified sample by longitude.
t0 = time.perf_counter()
coarse = downsample_bbox(df_wave, AOI_BBOX, COARSE_N)
t_coarse = time.perf_counter() - t0
print(f'E.3 coarse base built: {len(coarse):,} pts in {t_coarse:.1f}s')

def make_two_layer_fig(fine: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(go.Scattermap(  # base coarse layer
        lon=coarse['ShLongitude'], lat=coarse['ShLatitude'], mode='markers',
        marker=dict(size=2, color=coarse['WaveHeight'], colorscale='Viridis',
                    cmin=0, cmax=0.5, opacity=0.5),
        hoverinfo='skip', name='coarse'))
    fig.add_trace(go.Scattermap(  # fine high-res viewport layer
        lon=fine['ShLongitude'], lat=fine['ShLatitude'], mode='markers',
        marker=dict(size=4, color=fine['WaveHeight'], colorscale='Viridis',
                    cmin=0, cmax=0.5),
        text=fine['WaveHeight'].round(3), hoverinfo='text', name='fine'))
    fig.update_layout(map=dict(style='carto-positron', center=dict(lon=103.65, lat=1.26), zoom=11),
                      margin=dict(l=0, r=0, t=0, b=0), height=550, uirevision='static')
    return fig

init_fine = downsample_bbox(df_wave, AOI_BBOX, FINE_N)

app = Dash(__name__)
app.layout = html.Div([
    html.H4('E.3 Two-resolution: coarse base + debounced fine top'),
    html.Div(id='banner', style={'fontFamily':'monospace','padding':'4px'}),
    dcc.Store(id='rl-debounced'),
    dcc.Graph(id='g', figure=make_two_layer_fig(init_fine)),
])
app.clientside_callback(
    "function(rl){clearTimeout(window._dbnc3);return new Promise(r=>{window._dbnc3=setTimeout(()=>r(rl),250);});}",
    Output('rl-debounced', 'data'), Input('g', 'relayoutData'))

@app.callback(Output('g', 'figure'), Output('banner', 'children'),
              Input('rl-debounced', 'data'), prevent_initial_call=True)
def update_fine(rl):
    if not rl: return no_update, no_update
    t0 = time.perf_counter()
    fine = downsample_bbox(df_wave, AOI_BBOX, FINE_N)
    return make_two_layer_fig(fine), f'fine≈{len(fine):,}  ds={(time.perf_counter()-t0)*1000:.0f}ms'

# app.run(jupyter_mode='inline', port=PORT_BASE+11, debug=False)
record('E', 'two_resolution', coarse_n=COARSE_N, fine_n=FINE_N,
       coarse_build_s=t_coarse, smoothness='RECORD MANUALLY')

In [ ]:
# E.4 — Pre-tiled aggregations. Precompute top-N at 5 zoom levels; lookup is O(1).
ZOOM_LEVELS = [10, 11, 12, 13, 14]
TILE_CAP = 50_000

t0 = time.perf_counter()
# For the spike, treat each zoom level as one global aggregation rather than tiles.
tiles = {z: downsample_bbox(df_wave, AOI_BBOX, TILE_CAP // (15 - z)) for z in ZOOM_LEVELS}
t_pretile = time.perf_counter() - t0
for z, t in tiles.items():
    print(f'  z={z}: {len(t):,} pts')
print(f'E.4 pre-tile total: {t_pretile:.1f}s')

# Lookup callback would simply pick the right tile based on map zoom — no downsample work.
record('E', 'pretiled_zoom_levels', n_levels=len(ZOOM_LEVELS), pretile_s=t_pretile,
       lookup_latency_ms='~0 (O(1) dict lookup)', smoothness='EXPECTED smooth')

In [ ]:
# E.5 — Partial property updates via Patch().
# Update only trace lat/lon/marker.color arrays — not the whole figure.
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, Patch, no_update

init = downsample_bbox(df_wave, AOI_BBOX, MAX_POINTS)

app = Dash(__name__)
app.layout = html.Div([
    html.H4('E.5 Partial Patch() updates'),
    html.Div(id='banner', style={'fontFamily':'monospace','padding':'4px'}),
    dcc.Store(id='rl-debounced'),
    dcc.Graph(id='g', figure=make_fig(init)),
])
app.clientside_callback(
    "function(rl){clearTimeout(window._dbnc5);return new Promise(r=>{window._dbnc5=setTimeout(()=>r(rl),250);});}",
    Output('rl-debounced','data'), Input('g','relayoutData'))

@app.callback(Output('g','figure'), Output('banner','children'),
              Input('rl-debounced','data'), prevent_initial_call=True)
def patch_update(rl):
    if not rl: return no_update, no_update
    t0 = time.perf_counter()
    sub = downsample_bbox(df_wave, AOI_BBOX, MAX_POINTS)
    p = Patch()
    p['data'][0]['lon'] = sub['ShLongitude'].tolist()
    p['data'][0]['lat'] = sub['ShLatitude'].tolist()
    p['data'][0]['marker']['color'] = sub['WaveHeight'].tolist()
    p['data'][0]['text'] = sub['WaveHeight'].round(3).tolist()
    return p, f'patch sent: {len(sub):,} pts  ds={(time.perf_counter()-t0)*1000:.0f}ms'

# app.run(jupyter_mode='inline', port=PORT_BASE+12, debug=False)
record('E', 'patch_partial_update', smoothness='RECORD MANUALLY',
       json_savings='Patch transmits only changed arrays')

## Synthesis

Print the collected metrics and propose an architecture for the Dash app spec.

In [ ]:
df_results = pd.DataFrame(RESULTS)
with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(df_results.to_string(index=False))
df_results.to_csv(REPO / 'dash_rendering_spike_results.csv', index=False)
print('\nsaved → dash_rendering_spike_results.csv')

### Recommended architecture (fill in after running)

Based on measurements above, the Dash app spec should use:

- **Wave-impact map (state A):** ___ + bbox top-N (cap = ___ points) + debounced relayout (___ ms) + clientside hover-toggle at ___ visible-points threshold + always-on coarse base layer of ___ points.
- **Coastline strip (state B):** Scattergl + plotly-resampler `MinMaxLTTB`, `default_n_shown_samples = ___`.
- **Vessel + tracks dense overview (state C):** ___ (datashader image layer / dash-deck).
- **Filtered drill-down (state D):** ___ (Scattermap / dash-deck with picking).
- **Pre-tiled aggregations (E.4):** required / not required based on whether E.2 + E.3 alone hit the smoothness bar.